# Tutorial: Statistics, Probability & Linear Algebra

> **Type:** Tutorial — follow along step by step.  
> **Time:** ~2 hours  
> **Prerequisites:** Module 02, numpy basics  
> **What you'll build:** A complete mathematical toolkit: probability simulations, full statistical analysis of the iris dataset, and a from-scratch linear regression using only matrix operations.

---

## What We're Building

Three sections:
1. **Probability** — simulate coin flips, visualise distributions, demonstrate Bayes' theorem on a real disease-testing scenario
2. **Statistics** — full descriptive analysis of the iris dataset with correlation heatmap
3. **Linear algebra** — build linear regression from scratch using only `numpy` matrix operations, verify it matches sklearn

## Step 0: Setup

In [ ]:
# !pip install numpy scipy matplotlib seaborn scikit-learn

import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.linear_model import LinearRegression

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.rcParams['figure.figsize'] = (13, 5)
sns.set_style('whitegrid')
print('Setup complete.')

---

## Part 1: Probability

### Step 1: The Law of Large Numbers

Flip a fair coin many times. As the number of flips grows, the observed frequency of heads converges to the true probability 0.5. This is the **law of large numbers** — the empirical foundation of all probability theory.

In [ ]:
# Simulate coin flips
n_flips = 10_000
flips = np.random.binomial(1, p=0.5, size=n_flips)  # 1 = heads, 0 = tails

# Running average of heads
running_avg = np.cumsum(flips) / np.arange(1, n_flips + 1)

plt.figure(figsize=(10, 4))
plt.plot(running_avg, color='#4E79A7', linewidth=1, label='Observed frequency of heads')
plt.axhline(0.5, color='#E15759', linestyle='--', linewidth=1.5, label='True probability (0.5)')
plt.xscale('log')
plt.xlabel('Number of flips (log scale)')
plt.ylabel('Frequency of heads')
plt.title('Law of Large Numbers: Coin Flip Simulation')
plt.legend()
plt.ylim(0, 1)
plt.tight_layout()
plt.savefig('law_of_large_numbers.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"After {n_flips:,} flips: observed P(heads) = {running_avg[-1]:.4f}")

### Step 2: Common Distributions

Visualise the four distributions from `explanation.md` side by side.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# 1. Gaussian
x_gauss = np.linspace(-4, 4, 300)
for mu, sigma, label in [(0, 1, 'μ=0, σ=1'), (0, 2, 'μ=0, σ=2'), (1, 1, 'μ=1, σ=1')]:
    axes[0].plot(x_gauss, stats.norm.pdf(x_gauss, mu, sigma), label=label)
axes[0].set_title('Gaussian (Normal)')
axes[0].legend(fontsize=8)
axes[0].set_xlabel('x')
axes[0].set_ylabel('P(x)')

# 2. Bernoulli
for p, colour in [(0.3, '#4E79A7'), (0.5, '#F28E2B'), (0.7, '#E15759')]:
    axes[1].bar([0 - 0.1 + p*0.1, 1 - 0.1 + p*0.1], 
                [1-p, p], width=0.08, label=f'p={p}', alpha=0.8)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['0 (failure)', '1 (success)'])
axes[1].set_title('Bernoulli')
axes[1].legend(fontsize=8)
axes[1].set_ylabel('P(x)')

# 3. Binomial (N=20 trials)
N = 20
k = np.arange(0, N+1)
for p, colour in [(0.3, '#4E79A7'), (0.5, '#F28E2B'), (0.7, '#E15759')]:
    pmf = stats.binom.pmf(k, N, p)
    axes[2].plot(k, pmf, 'o-', label=f'p={p}', markersize=4)
axes[2].set_title(f'Binomial (N={N})')
axes[2].set_xlabel('k successes')
axes[2].legend(fontsize=8)
axes[2].set_ylabel('P(X=k)')

# 4. Poisson
k_poi = np.arange(0, 20)
for lam, colour in [(1, '#4E79A7'), (4, '#F28E2B'), (8, '#E15759')]:
    pmf = stats.poisson.pmf(k_poi, lam)
    axes[3].plot(k_poi, pmf, 'o-', label=f'λ={lam}', markersize=4)
axes[3].set_title('Poisson')
axes[3].set_xlabel('k events')
axes[3].legend(fontsize=8)
axes[3].set_ylabel('P(X=k)')

plt.suptitle('Common Probability Distributions', fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### Step 3: Bayes' Theorem — The Disease Test Example

From `explanation.md`: a disease has 1% prevalence. A test is 99% sensitive and 95% specific. If someone tests positive, what's the actual probability they have the disease?

In [ ]:
# Known quantities
P_disease       = 0.01   # prevalence
P_pos_given_dis = 0.99   # sensitivity (true positive rate)
P_neg_given_healthy = 0.95  # specificity (true negative rate)
P_pos_given_healthy = 1 - P_neg_given_healthy  # false positive rate
P_healthy       = 1 - P_disease

# Total probability of testing positive (Law of Total Probability)
P_positive = (P_pos_given_dis * P_disease) + (P_pos_given_healthy * P_healthy)

# Bayes' theorem: P(disease | positive)
P_disease_given_pos = (P_pos_given_dis * P_disease) / P_positive

print("=== Bayes' Theorem: Disease Test ===")
print(f"  Disease prevalence:          {P_disease:.0%}")
print(f"  Test sensitivity:            {P_pos_given_dis:.0%}")
print(f"  Test specificity:            {P_neg_given_healthy:.0%}")
print(f"  P(positive):                 {P_positive:.3f}")
print(f"  P(disease | test positive):  {P_disease_given_pos:.1%}")
print()
print("Despite a 99% accurate test, only 16.7% of positive tests are true positives!")
print("This is because the disease is rare — the base rate dominates.")

# Verify with simulation
population = 100_000
n_diseased = int(population * P_disease)
n_healthy  = population - n_diseased

# True positives and false positives
true_positives  = int(n_diseased * P_pos_given_dis)
false_positives = int(n_healthy  * P_pos_given_healthy)
all_positives   = true_positives + false_positives

print(f"\nSimulation over {population:,} people:")
print(f"  True positives:    {true_positives:,}")
print(f"  False positives:   {false_positives:,}")
print(f"  Empirical P(disease|pos): {true_positives/all_positives:.1%}")

> **Why this matters for ML:** Every time you evaluate a classifier on imbalanced data, you're doing Bayes' theorem implicitly. A model with 99% accuracy on 1% positive data may have terrible precision. The base rate (class imbalance) always matters.

---

## Part 2: Statistics on Real Data

### Step 4: Load the Iris Dataset

In [ ]:
# Load iris — the classic 4-feature, 3-class dataset
iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)

print(f"Shape: {df.shape}")
print(f"Species: {df['species'].unique()}")
df.head()

### Step 5: Descriptive Statistics

In [ ]:
# Full summary stats
desc = df.describe().round(3)
print(desc)

# Notice: mean vs median — are distributions skewed?
numeric_cols = iris.feature_names
print("\nMean vs Median (large difference = skewed distribution):")
for col in numeric_cols:
    mean   = df[col].mean()
    median = df[col].median()
    diff   = abs(mean - median)
    print(f"  {col:<30}: mean={mean:.2f}, median={median:.2f}, diff={diff:.3f}")

### Step 6: Distributions by Species

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
colours = {'setosa': '#4E79A7', 'versicolor': '#F28E2B', 'virginica': '#E15759'}

for ax, col in zip(axes, numeric_cols):
    for species, colour in colours.items():
        data = df[df['species'] == species][col]
        ax.hist(data, bins=15, alpha=0.6, label=species, color=colour, density=True)
        ax.axvline(data.mean(), color=colour, linestyle='--', linewidth=1.5)
    ax.set_title(col, fontsize=9)
    ax.set_xlabel('cm')
    ax.legend(fontsize=7)

axes[0].set_ylabel('Density')
plt.suptitle('Iris Feature Distributions by Species (dashed lines = means)', 
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('iris_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

### Step 7: Correlation Matrix

In [ ]:
corr_matrix = df[numeric_cols].corr()
print("Correlation matrix:")
print(corr_matrix.round(3))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Heatmap
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, ax=axes[0],
            square=True, linewidths=0.5)
axes[0].set_title('Pearson Correlation Matrix', fontweight='bold')

# Scatter: most correlated pair
axes[1].scatter(df['petal length (cm)'], df['petal width (cm)'],
                c=[{'setosa':0,'versicolor':1,'virginica':2}[s] for s in df['species']],
                cmap='Set1', alpha=0.7, s=50)
axes[1].set_xlabel('Petal length (cm)')
axes[1].set_ylabel('Petal width (cm)')
r = corr_matrix.loc['petal length (cm)', 'petal width (cm)']
axes[1].set_title(f'Most correlated pair\nr = {r:.3f}', fontweight='bold')

plt.tight_layout()
plt.savefig('iris_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

### Step 8: Normality Test

Many statistical tests assume Gaussian data. Test this assumption before applying them.

In [ ]:
print("Shapiro-Wilk test for normality (p < 0.05 → probably not Gaussian):\n")
for col in numeric_cols:
    stat, p = stats.shapiro(df[col])
    result = "✓ Gaussian" if p > 0.05 else "✗ Not Gaussian"
    print(f"  {col:<30}: W={stat:.4f}, p={p:.4f}  {result}")

print("\n(Iris mixes species, so the full distribution is multimodal — not Gaussian.)")
print("Per-species distributions are closer to Gaussian:")
for col in numeric_cols[:2]:
    setosa_data = df[df['species']=='setosa'][col]
    stat, p = stats.shapiro(setosa_data)
    print(f"  setosa {col:<25}: p={p:.4f}")

---

## Part 3: Linear Algebra — Linear Regression from Scratch

### Step 9: The Dataset and Goal

We'll predict petal width from petal length using the normal equations: **w = (X^T X)^{-1} X^T y**

In [ ]:
# Feature matrix X and target vector y
X_raw = df[['petal length (cm)']].values  # shape (150, 1)
y     = df['petal width (cm)'].values     # shape (150,)

# Add a bias column (column of 1s) — this gives us the intercept term
# X becomes shape (150, 2): [1, petal_length]
X = np.column_stack([np.ones(len(X_raw)), X_raw])

print(f"X shape: {X.shape}  — {X.shape[0]} data points, {X.shape[1]} features (bias + petal length)")
print(f"y shape: {y.shape}")
print(f"\nFirst 5 rows of X:")
print(X[:5])

### Step 10: Solve with the Normal Equations

In [ ]:
# Normal equations: w = (X^T X)^{-1} X^T y
# We use np.linalg.solve instead of direct inversion — numerically more stable
XtX = X.T @ X             # (2 x 150) @ (150 x 2) = (2 x 2)
Xty = X.T @ y             # (2 x 150) @ (150,)    = (2,)
w   = np.linalg.solve(XtX, Xty)  # solve XtX @ w = Xty

print(f"Learned weights:")
print(f"  Intercept (w₀): {w[0]:.4f}")
print(f"  Slope     (w₁): {w[1]:.4f}")
print(f"\nModel: petal_width = {w[0]:.4f} + {w[1]:.4f} × petal_length")

### Step 11: Verify Against sklearn

In [ ]:
# sklearn's LinearRegression — should match exactly
lr = LinearRegression()
lr.fit(X_raw, y)

print("Comparison: numpy normal equations vs sklearn LinearRegression")
print(f"  Intercept — numpy: {w[0]:.6f}   sklearn: {lr.intercept_:.6f}")
print(f"  Slope     — numpy: {w[1]:.6f}   sklearn: {lr.coef_[0]:.6f}")

match = np.allclose([w[0], w[1]], [lr.intercept_, lr.coef_[0]], atol=1e-6)
print(f"\nResults match: {match}")

### Step 12: Visualise the Fit

In [ ]:
# Predictions
y_hat = X @ w  # matrix multiplication: shape (150,)
residuals = y - y_hat
mse = np.mean(residuals**2)
rmse = np.sqrt(mse)
ss_res = np.sum(residuals**2)
ss_tot = np.sum((y - y.mean())**2)
r_squared = 1 - ss_res / ss_tot

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter with fit line
x_line = np.array([X_raw.min(), X_raw.max()])
y_line = w[0] + w[1] * x_line

axes[0].scatter(X_raw, y, alpha=0.5, color='#4E79A7', s=30, label='Data')
axes[0].plot(x_line, y_line, color='#E15759', linewidth=2, label=f'Fit: R²={r_squared:.3f}')
axes[0].set_xlabel('Petal length (cm)')
axes[0].set_ylabel('Petal width (cm)')
axes[0].set_title('Linear Regression: From-Scratch Normal Equations')
axes[0].legend()

# Residuals plot
axes[1].scatter(y_hat, residuals, alpha=0.5, color='#F28E2B', s=30)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xlabel('Predicted petal width')
axes[1].set_ylabel('Residual (actual − predicted)')
axes[1].set_title(f'Residuals Plot\nRMSE = {rmse:.4f} cm')

plt.tight_layout()
plt.savefig('linear_regression_scratch.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"R² = {r_squared:.4f}  (proportion of variance explained)")
print(f"RMSE = {rmse:.4f} cm  (average prediction error)")

> **What just happened?** You implemented linear regression using nothing but matrix operations — the same math that sklearn computes in `lr.fit()`. Every supervised ML model is, at its core, learning to minimise a loss function. Linear regression minimises the squared difference between predictions and reality, and it has a closed-form solution. Most models don't — that's why we need gradient descent (Module 04).

---

## What You Built

A complete mathematical toolkit: you simulated and visualised probability distributions, applied Bayes' theorem to understand why base rates matter in imbalanced classification, ran a full statistical analysis on a real dataset, and implemented linear regression from scratch using matrix operations — verifying it matches sklearn exactly.

## What's Next

- Read `explanation.md` for the conceptual framing of why these tools matter in ML.
- Keep `reference.md` open — the distribution cheat sheet and linear algebra operations table will be useful in every future module.
- **Extension challenge:** Extend the linear regression to multiple features (petal length + sepal length). You only need to change the X matrix — the normal equations stay the same.

**Next module:** `04_math_calculus_optimization` — why most models can't use the normal equations and need gradient descent instead.

---

**AWS SageMaker note:** Statistical analysis at scale (millions of rows) is the province of SageMaker Data Wrangler, which runs these computations on managed infrastructure. In Module 26 we'll use it. Here, pandas + scipy on a local machine is the right tool.